# LangGraph Exercise: Adding Simple Routing
## Workshop Section 3 - Part 5

**Objective**: Implement routing logic to create a stop condition for the LangGraph agent loop.

**Learning Goals**:
- Understand how to implement conditional routing in LangGraph
- Learn to use `Command` for dynamic edge transitions
- Practice state inspection with `get_nth_message`
- Create a conversation loop with proper termination

## Setup and Imports

In [ ]:
import uuid
from typing import Annotated, Optional, Literal
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command
from langgraph.graph.message import add_messages
from functools import partial
from colorama import Fore, Style
from copy import deepcopy
import operator

## LLM Configuration

In [ ]:
from langchain_nvidia import ChatNVIDIA

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", base_url="http://llm_client:9000/v1")

## State Definition

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    interactions: Annotated[int, operator.__add__]
    extra_kwargs: Optional[dict]

## Helper Functions

In [ ]:
def get_nth_message(state: State, n=-1, attr="messages"):
    """Retrieve the nth message from the state's message list."""
    try: 
        return state.get("messages")[n].content
    except: 
        return ""

## Node Functions

In [ ]:
def user(state: State):
    """User input node that interrupts execution to collect user input."""
    answer = interrupt("[User]:")
    return {"messages": [("user", answer)]}

In [ ]:
def agent(state: State, config=None):
    """Agent response node that invokes the LLM to generate a response."""
    response = llm.invoke(state.get("messages"), config=config)
    return {"messages": [response]}

In [ ]:
def route(state: State, config=None):
    """
    Routing node - EXERCISE FUNCTION.
    Returns Command to END if 'stop' detected, otherwise routes to 'user'.
    """
    last_message = get_nth_message(state, n=-1)
    
    if "stop" in last_message.lower():
        return Command(update={"interactions": 1}, goto=END)
    else:
        return Command(update={"interactions": 1}, goto="user")

## Graph Construction

In [ ]:
builder = StateGraph(State)
builder.add_edge(START, "user")
builder.add_node("agent", agent)
builder.add_node("user", user)
builder.add_node("route", route)
builder.add_edge("user", "agent")
builder.add_edge("agent", "route")

## Compilation and Configuration

In [ ]:
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

config = {
    "configurable": {
        "thread_id": uuid.uuid4(),
    }
}

app_stream = partial(app.stream, config=config)

## Streaming Utility Function

In [ ]:
def stream_from_app_simple(app_stream, input_buffer=[{"messages": []}], verbose=False, debug=False):
    """Execute the agent system in a streaming fashion with interrupt handling."""
    seen_metas = dict()
    input_buffer = deepcopy(input_buffer)
    
    while input_buffer:
        for mode, chunk in app_stream(input_buffer.pop(), stream_mode=["values", "messages", "updates", "debug"]):
            if mode == "messages":
                chunk, meta = chunk
                if meta.get("checkpoint_ns") not in seen_metas:
                    caller_node = meta.get("langgraph_node")
                    yield f"[{caller_node.title()}]: "
                seen_metas[meta.get("checkpoint_ns")] = meta
                if chunk.content:
                    yield chunk.content
            elif mode == "updates":
                global v
                v = chunk
                if "__interrupt__" in chunk:
                    user_input = input("\n[Interrupt] " + chunk.get("__interrupt__")[0].value)
                    input_buffer.append(Command(resume=user_input))

## Exercise Execution

In [ ]:
print("=" * 60)
print("LangGraph Routing Exercise - Chat Interface")
print("Type 'stop' in any message to end the conversation")
print("=" * 60)
print()

for token in stream_from_app_simple(app_stream, verbose=False, debug=False):
    print(token, end="", flush=True)

print()
print("Conversation ended!")